In [1]:
# Core RAG Frameworks
!pip install langgraph
!pip install langchain-openai
!pip install langchain-community
!pip install langchain
!pip install tiktoken
!pip install openai
!pip install grandalf




In [2]:



# Import all depedencies

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,AIMessage,BaseMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt


# Import the Python SDK for Colab secrets
from google.colab import userdata
from typing import TypedDict, Annotated


In [60]:
import os

# Fetch the API key from Colab secrets
openai_api_key = userdata.get('OPENAI_APIKEY')

# Set up Environmental Variables
os.environ["OPENAI_API_KEY"] = openai_api_key

In [61]:
# ==========================================
# Step 1: Initialize LLM
# ==========================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [71]:
# ==========================================
# Step 2: Define State
# ==========================================

class ChatBotState(TypedDict) :
  msg : Annotated[list[BaseMessage], add_messages]

In [72]:
def chatbot_node(state: ChatBotState):
    verdict = interrupt({
        "type": "approval",
        "reason": "Model to answer questions",
        "question": state["msg"][-1].content,
        "instruction": "Answer this question with Yes/No"
    })

    if verdict.get("approved") == 'no':
        return {"msg": [AIMessage(content="Not Approved")]}
    elif verdict.get("approved") == 'yes':
        final_response = llm.invoke(state["msg"])
        return {"msg": [final_response]}
    else:
        return {"msg": [AIMessage(content="Answer yes/no only")]}

In [73]:
builder = StateGraph(ChatBotState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [74]:
# ============================================================================
# PART 6: Setup config
# ============================================================================

configSetUp = {"configurable": {"thread_id": "demo-thread"}}


In [75]:
# ============================================================================
# PART 6: Setup userQuery
# ============================================================================

user_input = {"msg": [HumanMessage(content="What is the capital of France?")]}



In [76]:
# ============================================================================
# PART 7: Fetch values
# ============================================================================

for event in graph.stream(user_input, config=configSetUp):
    print(event)

{'__interrupt__': (Interrupt(value={'type': 'approval', 'reason': 'Model to answer questions', 'question': 'What is the capital of France?', 'instruction': 'Answer this question with Yes/No'}, id='3fe49b37464262faf9244944ab993cd7'),)}


In [77]:
# ============================================================================
# PART 8: Current values of State when the graph is paused
# ============================================================================

# Read the current paused state payload
current_state = graph.get_state(configSetUp)
print(current_state.tasks[0].interrupts[0].value)

{'type': 'approval', 'reason': 'Model to answer questions', 'question': 'What is the capital of France?', 'instruction': 'Answer this question with Yes/No'}


In [78]:
# ============================================================================
# PART 9: Get the value from user to continue the graph or not
# ============================================================================

user_input = input("Should LLM answer this question or not , say only yes or no ==> ")

Should LLM answer this question or not , say only yes or no ==> yes


In [79]:
# ============================================================================
# PART 9: Resume values of State when the graph is paused
# ============================================================================

for event in graph.stream(Command(resume={"approved": user_input}), config=configSetUp):
    print(event['chatbot']['msg'][0].content)

The capital of France is Paris.
